In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchaudio")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torchaudio import datasets, transforms
import random

In [3]:
COMMANDS_10 = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
LABELS = COMMANDS_10 + ['unknown', 'silence']

labelToIdx = {l: i for i, l in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)  # 12

In [4]:
mel_spec = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=512,
    hop_length=160,
    win_length=400,
    n_mels=64,
    f_min=0.0,
    f_max=8000.0,
)

In [5]:
TARGET_LENGTH = 16000

def collate_fn(batch, training=False):
    waveforms, labels = [], []

    for waveform, sample_rate, label, *_ in batch:
        # Pad/trim
        if waveform.shape[-1] < TARGET_LENGTH:
            waveform = F.pad(waveform, (0, TARGET_LENGTH - waveform.shape[-1]))
        else:
            waveform = waveform[..., :TARGET_LENGTH]

        # Time shift augmentation (ONLY during training)
        if training:
            shift = random.randint(-800, 800)
            if shift > 0:
                waveform = F.pad(waveform[..., :-shift], (shift, 0))
            elif shift < 0:
                waveform = F.pad(waveform[..., -shift:], (0, -shift))
        if label in COMMANDS_10:
            mapped_label = labelToIdx[label]
        else:
            mapped_label = labelToIdx['unknown']

        waveforms.append(waveform)
        labels.append(mapped_label)

    # Add silence samples (~10%)
    if training:
        num_silence = int(0.1 * len(waveforms))
        for _ in range(num_silence):
            waveforms.append(torch.zeros(1, TARGET_LENGTH))
            labels.append(labelToIdx['silence'])

    return torch.stack(waveforms), torch.tensor(labels)

In [6]:
train_collate = lambda batch: collate_fn(batch, training=True)
test_collate  = lambda batch: collate_fn(batch, training=False)

In [7]:
SC_train = datasets.SPEECHCOMMANDS(root='data', subset='training', download=True, url='speech_commands_v0.02')
SC_val   = datasets.SPEECHCOMMANDS(root='data', subset='validation', download=True, url='speech_commands_v0.02')
SC_test  = datasets.SPEECHCOMMANDS(root='data', subset='testing',  download=True, url='speech_commands_v0.02')

In [8]:
train_loader = torch.utils.data.DataLoader(SC_train, batch_size=128, num_workers=4, pin_memory=True, shuffle=True, collate_fn=train_collate)
val_loader = torch.utils.data.DataLoader(SC_val, batch_size=1024, num_workers=4, pin_memory=True, shuffle=False, collate_fn=test_collate)
test_loader = torch.utils.data.DataLoader(SC_test, batch_size=1024, num_workers=4, pin_memory=True, shuffle=False, collate_fn=test_collate)

In [9]:
class KeywordGRU(nn.Module):
    def __init__(self, n_mels=64, hidden_size=128, num_layers=2, num_classes=NUM_CLASSES):
        super().__init__()
        self.mel = mel_spec
        self.db = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=6)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=12)
        
        self.gru = nn.GRU(
            input_size=n_mels,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.mel(x)        # (batch, 1, n_mels, time)
        x = self.db(x)         # log-mel
        x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-5)

        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)
        x = x.squeeze(1)       # (batch, n_mels, time)
        x = x.permute(0, 2, 1) # (batch, time, n_mels)

        out, _ = self.gru(x)
        weights = torch.softmax(self.attention(out), dim=1)  # (batch, time, 1)
        out = (out * weights).sum(dim=1)  # (batch, hidden)
        return self.classifier(out)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KeywordGRU().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=7e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Device: cuda
GPU Name: NVIDIA H200 MIG 2g.35gb
GPU Memory: 34.90 GB
Total model parameters: 191,757
Trainable parameters: 191,757


In [11]:
def train(epoch=None):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        preds = output.argmax(dim=1)
        correct += (preds == target).sum().item()
        total += target.size(0)

        if batch_idx % 500 == 0 and epoch is not None:
            print(f"Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")

    avg_loss = total_loss / len(train_loader)

    accuracy = 100. * correct / total

    if epoch is not None:
        print(f"Epoch {epoch} - Avg Loss: {avg_loss:.6f} | Train Acc: {accuracy:.2f}%")

    return accuracy

In [12]:
def evaluate(loader, split_name="Val"):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item() * target.size(0)
            preds = output.argmax(dim=1)
            correct += (preds == target).sum().item()
            total += target.size(0)

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    print(f"{split_name}: loss={avg_loss:.4f}, acc={correct}/{total} ({acc:.2f}%)")
    return acc, avg_loss

In [ ]:
best_val = -1.0
bad_epochs = 0
patience = 5
EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    train_acc = train(epoch)
    val_acc, val_loss = evaluate(val_loader, "Val")
    scheduler.step(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        bad_epochs = 0
        torch.save(model.state_dict(), "best_keyword_gru.pt")
        print(f"saved best: {best_val:.2f}%")
    else:
        bad_epochs += 1
        print(f"no improvement: {bad_epochs}/{patience}")

    if bad_epochs >= patience:
        print("early stopping")
        break

Epoch 1 [0/84843] Loss: 2.511429


In [ ]:
model.load_state_dict(torch.load("best_keyword_gru.pt", map_location=device))
evaluate(test_loader, "Test")